In [5]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow import keras
from keras.layers import TextVectorization, Embedding, GlobalAveragePooling1D, Dense
from keras import Sequential

from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
tf.random.set_seed(42)

In [6]:
# One-time downloads for nltk's tokenizer, stopword list, and lemmatizer dictionary

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [7]:
df = pd.read_csv('/content/smartcart_customer_queries.csv')
print(df.shape)
df.head(10)

(420, 3)


,query_id,query_text,category
0,1,kids dress,Apparel
1,2,Organic Sugar Nestle,Grocery
2,3,KIDS DRESS GREY SIZE 8,Apparel
3,4,SUGAR FREE BISCUITS,Grocery
4,5,sugar free peanut butter,Grocery
5,6,headphones,Electronics
6,7,GAMING MOUSE BELOW 5000,Electronics
7,8,gaming mouse between 3000 and 6000...,Electronics
8,9,Wireless Earbuds Boat,Electronics
9,10,waterproof power bank JBL,Electronics


In [8]:
df['category'].value_counts()

,count
category,
Apparel,140
Grocery,140
Electronics,140


In [9]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['query_text'].apply(clean_text)
df[['query_text', 'clean_text']].sample(5)

,query_text,clean_text
145,delivery was late but the smartwatch quality i...,delivery was late but the smartwatch quality i...
334,men's party wear gown navy blue,men s party wear gown navy blue
175,wireless charger cable,wireless charger cable
369,latest smart bulb,latest smart bulb
416,best wifi router between 3000 and 6000,best wifi router between 3000 and 6000


In [10]:
df['tokens'] = df['clean_text'].apply(word_tokenize)
df[['clean_text', 'tokens']].sample(5, random_state=1)

,clean_text,tokens
378,packaging was damaged but the sneakers itself ...,"[packaging, was, damaged, but, the, sneakers, ..."
165,1kg pack organic honey aashirvaad,"[1kg, pack, organic, honey, aashirvaad]"
17,hoodie,[hoodie]
179,gluten free spices combo nestle,"[gluten, free, spices, combo, nestle]"
4,sugar free peanut butter,"[sugar, free, peanut, butter]"


In [11]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

df['tokens_no_stop'] = df['tokens'].apply(remove_stopwords)
df[['tokens', 'tokens_no_stop']].sample(5, random_state=1)

,tokens,tokens_no_stop
378,"[packaging, was, damaged, but, the, sneakers, ...","[packaging, damaged, sneakers, fine]"
165,"[1kg, pack, organic, honey, aashirvaad]","[1kg, pack, organic, honey, aashirvaad]"
17,[hoodie],[hoodie]
179,"[gluten, free, spices, combo, nestle]","[gluten, free, spices, combo, nestle]"
4,"[sugar, free, peanut, butter]","[sugar, free, peanut, butter]"


In [12]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

sample_words = ['running', 'shoes', 'organic', 'cheaper', 'earbuds', 'jackets']

comparison = pd.DataFrame({
    'word': sample_words,
    'stemmed': [stemmer.stem(w) for w in sample_words],
    'lemmatized': [lemmatizer.lemmatize(w) for w in sample_words]})
comparison

,word,stemmed,lemmatized
0,running,run,running
1,shoes,shoe,shoe
2,organic,organ,organic
3,cheaper,cheaper,cheaper
4,earbuds,earbud,earbuds
5,jackets,jacket,jacket


In [13]:
def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

df['final_tokens'] = df['tokens_no_stop'].apply(lemmatize_tokens)
df['processed_text'] = df['final_tokens'].apply(lambda toks: ' '.join(toks))
df[['query_text', 'processed_text']].sample(8, random_state=1)

,query_text,processed_text
378,packaging was damaged but the sneakers itself ...,packaging damaged sneaker fine
165,1KG PACK ORGANIC HONEY AASHIRVAAD,1kg pack organic honey aashirvaad
17,hoodie,hoodie
179,GLUTEN FREE SPICES COMBO NESTLE,gluten free spice combo nestle
4,sugar free peanut butter,sugar free peanut butter
249,Unpolished Organic Honey Nestle,unpolished organic honey nestle
213,natural organic honey,natural organic honey
361,best smartphone!!,best smartphone


In [14]:
sample_raw = df['query_text'].iloc[10]
print('1. Raw:       ', repr(sample_raw))
print('2. Cleaned:   ', clean_text(sample_raw))
toks = word_tokenize(clean_text(sample_raw))
print('3. Tokenized: ', toks)
toks_ns = remove_stopwords(toks)
print('4. No stopwords:', toks_ns)
toks_lem = lemmatize_tokens(toks_ns)
print('5. Lemmatized:', toks_lem)

1. Raw:        'Smartphone Under 2000!!'
2. Cleaned:    smartphone under 2000
3. Tokenized:  ['smartphone', 'under', '2000']
4. No stopwords: ['smartphone', '2000']
5. Lemmatized: ['smartphone', '2000']


In [15]:
processed_texts = df['processed_text'].tolist()

bow_vectorizer = TextVectorization(output_mode='count', max_tokens=2000)
bow_vectorizer.adapt(processed_texts)                      # builds the vocabulary

vocab = bow_vectorizer.get_vocabulary()
print('Vocabulary size:', len(vocab))

Vocabulary size: 182


In [16]:
sample_queries = [processed_texts[10], processed_texts[50]]

for text in sample_queries:
    vec = bow_vectorizer([text])[0].numpy()
    nonzero = {vocab[i]: int(vec[i]) for i in np.nonzero(vec)[0]}
    print(f'Query: "{text}"  ->  BoW: {nonzero}')

Query: "smartphone 2000"  ->  BoW: {np.str_('2000'): 1, np.str_('smartphone'): 1}
Query: "power bank"  ->  BoW: {np.str_('power'): 1, np.str_('bank'): 1}


In [17]:
tfidf_vectorizer = TextVectorization(output_mode='tf_idf', max_tokens=2000)
tfidf_vectorizer.adapt(processed_texts)

tfidf_vocab = tfidf_vectorizer.get_vocabulary()

In [18]:
# Compare a common word vs. a rare word directly: how many queries each appears in,
# and the tf-idf weight that a single occurrence of that word gets.
common_word, rare_word = 'shoe', 'basmati'

for w in [common_word, rare_word]:
    if w in tfidf_vocab:
        doc_freq = sum(1 for t in processed_texts if w in t.split())
        tfidf_col = tfidf_vocab.index(w)
        single_word_weight = tfidf_vectorizer([w])[0, tfidf_col].numpy()
        print(f"'{w}': appears in {doc_freq} of {len(processed_texts)} queries "
              f"-> tf-idf weight per occurrence ~ {single_word_weight:.3f}")
    else:
        print(f"'{w}' not in vocabulary -- pick another word from `vocab` to compare")

'shoe': appears in 16 of 420 queries -> tf-idf weight per occurrence ~ 3.247
'basmati': appears in 5 of 420 queries -> tf-idf weight per occurrence ~ 4.263


In [19]:
sequence_length = int(np.percentile(df['processed_text'].str.split().apply(len), 95))
print('Chosen padding length (95th percentile of query length):', sequence_length)

int_vectorizer = TextVectorization(
    output_mode='int', max_tokens=2000, output_sequence_length=sequence_length)
int_vectorizer.adapt(processed_texts)

X_sequences = int_vectorizer(np.array(processed_texts)).numpy()
print('Padded sequence matrix shape:', X_sequences.shape)
print('\nExample:')
print('  text:    ', processed_texts[10])
print('  sequence:', X_sequences[10])

Chosen padding length (95th percentile of query length): 7
Padded sequence matrix shape: (420, 7)

Example:
  text:     smartphone 2000
  sequence: [85 74  0  0  0  0  0]


In [20]:
# Labels, ready for training a model on top of these sequences
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['category'])
class_names = label_encoder.classes_
vocab_size = int_vectorizer.vocabulary_size()
print('Vocabulary size for embeddings:', vocab_size)
print('Classes:', class_names)

Vocabulary size for embeddings: 183
Classes: ['Apparel' 'Electronics' 'Grocery']


In [21]:
embedding_dim = 16

embed_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=sequence_length),
    GlobalAveragePooling1D(),          # average the word vectors in each query into one vector
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])

embed_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
metrics=['accuracy'])
embed_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
history = embed_model.fit(X_sequences, y, epochs=30, validation_split=0.2, verbose=0)

print('Final training accuracy:  ', round(history.history['accuracy'][-1], 3))
print('Final validation accuracy:', round(history.history['val_accuracy'][-1], 3))

Final training accuracy:   1.0
Final validation accuracy: 1.0


In [23]:
embedding_weights = embed_model.layers[0].get_weights()[0]   # shape: (vocab_size, embedding_dim)
embed_vocab = int_vectorizer.get_vocabulary()

def word_vector(word):
    if word not in embed_vocab:
        return None
    return embedding_weights[embed_vocab.index(word)]

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

pairs = [('shoe', 'sneaker'), ('cheap', 'budget'), ('organic', 'premium'),
         ('rice', 'sugar'), ('shoe', 'rice')]

for w1, w2 in pairs:
    v1, v2 = word_vector(w1), word_vector(w2)
    if v1 is not None and v2 is not None:
        print(f"cosine_similarity('{w1}', '{w2}') = {cosine_similarity(v1, v2):.3f}")
    else:
        print(f"'{w1}' or '{w2}' not in vocabulary -- try another pair from your dataset")

cosine_similarity('shoe', 'sneaker') = 0.983
cosine_similarity('cheap', 'budget') = 0.881
cosine_similarity('organic', 'premium') = 0.978
cosine_similarity('rice', 'sugar') = 0.965
cosine_similarity('shoe', 'rice') = -0.412


In [24]:
embedding_weights = embed_model.layers[0].get_weights()[0]   # shape: (vocab_size, embedding_dim)
embed_vocab = int_vectorizer.get_vocabulary()

def word_vector(word):
    if word not in embed_vocab:
        return None
    return embedding_weights[embed_vocab.index(word)]

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

pairs = [('shoe', 'sneaker'), ('cheap', 'budget'), ('organic', 'premium'),
         ('rice', 'sugar'), ('shoe', 'rice')]

for w1, w2 in pairs:
    v1, v2 = word_vector(w1), word_vector(w2)
    if v1 is not None and v2 is not None:
        print(f"cosine_similarity('{w1}', '{w2}') = {cosine_similarity(v1, v2):.3f}")
    else:
        print(f"'{w1}' or '{w2}' not in vocabulary -- try another pair from your dataset")

cosine_similarity('shoe', 'sneaker') = 0.983
cosine_similarity('cheap', 'budget') = 0.881
cosine_similarity('organic', 'premium') = 0.978
cosine_similarity('rice', 'sugar') = 0.965
cosine_similarity('shoe', 'rice') = -0.412


In [25]:
embedding_weights = embed_model.layers[0].get_weights()[0]   # shape: (vocab_size, embedding_dim)
embed_vocab = int_vectorizer.get_vocabulary()

def word_vector(word):
    if word not in embed_vocab:
        return None
    return embedding_weights[embed_vocab.index(word)]

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

pairs = [('shoe', 'sneaker'), ('cheap', 'budget'), ('organic', 'premium'),
         ('rice', 'sugar'), ('shoe', 'rice')]

for w1, w2 in pairs:
    v1, v2 = word_vector(w1), word_vector(w2)
    if v1 is not None and v2 is not None:
        print(f"cosine_similarity('{w1}', '{w2}') = {cosine_similarity(v1, v2):.3f}")
    else:
        print(f"'{w1}' or '{w2}' not in vocabulary -- try another pair from your dataset")

cosine_similarity('shoe', 'sneaker') = 0.983
cosine_similarity('cheap', 'budget') = 0.881
cosine_similarity('organic', 'premium') = 0.978
cosine_similarity('rice', 'sugar') = 0.965
cosine_similarity('shoe', 'rice') = -0.412


In [26]:
new_query = "  Best WIRELESS Earbuds under 1500!! "

cleaned = clean_text(new_query)
tokens = word_tokenize(cleaned)
tokens_ns = remove_stopwords(tokens)
tokens_lem = lemmatize_tokens(tokens_ns)
processed = ' '.join(tokens_lem)

int_seq = int_vectorizer([processed])
predicted_probs = embed_model.predict(int_seq, verbose=0)[0]
predicted_class = class_names[np.argmax(predicted_probs)]

print('1. Raw:        ', repr(new_query))
print('2. Cleaned:    ', cleaned)
print('3. Tokens:     ', tokens)
print('4. No stopword:', tokens_ns)
print('5. Lemmatized: ', tokens_lem)
print('9. Padded seq: ', int_seq.numpy()[0])
print('10. Predicted category:', predicted_class)

1. Raw:         '  Best WIRELESS Earbuds under 1500!! '
2. Cleaned:     best wireless earbuds under 1500
3. Tokens:      ['best', 'wireless', 'earbuds', 'under', '1500']
4. No stopword: ['best', 'wireless', 'earbuds', '1500']
5. Lemmatized:  ['best', 'wireless', 'earbuds', '1500']
9. Padded seq:  [102  25 158   1   0   0   0]
10. Predicted category: Electronics


Lab no 4

In [27]:



import re
import numpy as np, pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow import keras
from keras.layers import TextVectorization, Embedding, SimpleRNN, Dense
from keras import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_SEED = 42
DATA_PATH = "/content/smartcart_customer_queries.csv"
MAX_TOKENS = 2000
EMBEDDING_DIM = 16
RNN_UNITS = 16
TEST_SIZE = 0.2
EPOCHS = 30
BATCH_SIZE = 32

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

nltk.download("punkt"); nltk.download("punkt_tab")
nltk.download("stopwords"); nltk.download("wordnet"); nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess(text: str) -> str:
    """Clean, tokenize, drop stopwords, and lemmatize one query."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df = pd.read_csv(DATA_PATH)
df['processed_text'] = df['query_text'].apply(preprocess)

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=df['category']
)

sequence_length = int(np.percentile(
    train_df['processed_text'].str.split().apply(len), 95))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [28]:

int_vectorizer = TextVectorization(
    output_mode='int',
    max_tokens=MAX_TOKENS,
    output_sequence_length=sequence_length
)
int_vectorizer.adapt(train_df['processed_text'].tolist())  # TRAIN text only

label_encoder = LabelEncoder().fit(df['category'])
class_names = label_encoder.classes_

X_train = int_vectorizer(train_df['processed_text'].tolist()).numpy()
y_train = label_encoder.transform(train_df['category'])
X_test = int_vectorizer(test_df['processed_text'].tolist()).numpy()
y_test = label_encoder.transform(test_df['category'])
vocab_size = int_vectorizer.vocabulary_size()

print(f"Train/test split: {len(train_df)} / {len(test_df)} queries")
print(f"Sequence length (training cutoff): {sequence_length}")
print(f"Vocabulary size (fit on train only): {vocab_size}")

Train/test split: 336 / 84 queries
Sequence length (training cutoff): 7
Vocabulary size (fit on train only): 182


In [29]:
def build_rnn_model(vocab_size: int, num_classes: int) -> keras.Model:
    """Embedding -> SimpleRNN -> Dense classifier for query intent."""
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM),
        SimpleRNN(RNN_UNITS),
        Dense(num_classes, activation='softmax')])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    return model

rnn_model = build_rnn_model(vocab_size, num_classes=len(class_names))
rnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True)

history = rnn_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=0)

test_loss, test_accuracy = rnn_model.evaluate(X_test, y_test, verbose=0)
print(f"Stopped after {len(history.history['loss'])} epochs "
      f"(best weights restored)")
print(f"Held-out test accuracy on short queries: {test_accuracy:.3f}")

Stopped after 16 epochs (best weights restored)
Held-out test accuracy on short queries: 0.976


In [31]:
long_query_samples = [
    {"ticket_id": "SC-4831-01", "expected_category": "Electronics",
     "query": "looking for a lightweight wireless bluetooth headphone "
              "with good battery life and noise cancelling under budget "
              "two thousand rupees with fast charging support"},
    {"ticket_id": "SC-4831-02", "expected_category": "Electronics",
     "query": "need a budget smartphone with good camera long battery "
              "backup fast charging support and at least six gb ram "
              "under fifteen thousand rupees"},
    {"ticket_id": "SC-4831-03", "expected_category": "Apparel",
     "query": "want a slim fit cotton casual shirt for men in navy blue "
              "color size large with full sleeves for daily office wear"},
    {"ticket_id": "SC-4831-04", "expected_category": "Apparel",
     "query": "looking for comfortable running shoes for women with good "
              "grip lightweight sole and breathable material for daily "
              "gym and outdoor jogging"},
    {"ticket_id": "SC-4831-05", "expected_category": "Grocery",
     "query": "want organic basmati rice five kilogram pack with long "
              "grain good aroma and no added chemicals for daily home "
              "cooking"},
    {"ticket_id": "SC-4831-06", "expected_category": "Grocery",
     "query": "looking for cold pressed olive oil one litre bottle with "
              "rich flavor and no preservatives for healthy daily cooking"},]

In [32]:
def evaluate_on_queries(model, vectorizer, samples):
    """Run raw queries through preprocessing + vectorizer + model,
    returning a results table with predictions and confidence."""
    texts = [preprocess(s['query']) for s in samples]
    sequences = vectorizer(texts).numpy()
    probs = model.predict(sequences, verbose=0)
    results = pd.DataFrame(samples)
    results['predicted_category'] = class_names[np.argmax(probs, axis=1)]
    results['confidence'] = probs.max(axis=1).round(2)
    return results

# Re-vectorize without the 7-token training cutoff -- these queries run
# well past it, and we want to see how the RNN handles the full length.
eval_vectorizer = TextVectorization(output_mode='int', max_tokens=MAX_TOKENS)
eval_vectorizer.adapt(train_df['processed_text'].tolist())  # same vocab

results_df = evaluate_on_queries(rnn_model, eval_vectorizer,
                                  long_query_samples)
long_query_accuracy = (
    results_df['predicted_category'] == results_df['expected_category']
).mean()

print(results_df[['ticket_id', 'expected_category',
                   'predicted_category', 'confidence']])
print(f"\nAccuracy on {len(long_query_samples)} queries from SC-4831: "
      f"{long_query_accuracy:.2f}")

    ticket_id expected_category predicted_category  confidence
0  SC-4831-01       Electronics            Grocery        0.74
1  SC-4831-02       Electronics            Grocery        0.96
2  SC-4831-03           Apparel        Electronics        0.63
3  SC-4831-04           Apparel            Apparel        0.95
4  SC-4831-05           Grocery            Apparel        0.94
5  SC-4831-06           Grocery            Apparel        0.89

Accuracy on 6 queries from SC-4831: 0.17


Lab no 5


In [33]:


import re
import numpy as np, pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow import keras
from keras.layers import TextVectorization, Embedding, LSTM, GRU, Dense
from keras import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_SEED = 42
DATA_PATH = "smartcart_customer_queries.csv"
MAX_TOKENS = 2000
EMBEDDING_DIM = 16
RNN_UNITS = 16
TEST_SIZE = 0.2
EPOCHS = 30
BATCH_SIZE = 32

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

nltk.download("punkt"); nltk.download("punkt_tab")
nltk.download("stopwords"); nltk.download("wordnet"); nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess(text: str) -> str:
    """Clean, tokenize, drop stopwords, and lemmatize one query."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df = pd.read_csv(DATA_PATH)
df['processed_text'] = df['query_text'].apply(preprocess)

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=df['category'])

sequence_length = int(np.percentile(
    train_df['processed_text'].str.split().apply(len), 95))

int_vectorizer = TextVectorization(
    output_mode='int', max_tokens=MAX_TOKENS, output_sequence_length=sequence_length)
int_vectorizer.adapt(train_df['processed_text'].tolist())

import re
import numpy as np, pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow import keras
from keras.layers import TextVectorization, Embedding, LSTM, GRU, Dense
from keras import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_SEED = 42
DATA_PATH = "smartcart_customer_queries.csv"
MAX_TOKENS = 2000
EMBEDDING_DIM = 16
RNN_UNITS = 16
TEST_SIZE = 0.2
EPOCHS = 30
BATCH_SIZE = 32

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

nltk.download("punkt"); nltk.download("punkt_tab")
nltk.download("stopwords"); nltk.download("wordnet"); nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess(text: str) -> str:
    """Clean, tokenize, drop stopwords, and lemmatize one query."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df = pd.read_csv(DATA_PATH)
df['processed_text'] = df['query_text'].apply(preprocess)

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=df['category'])

sequence_length = int(np.percentile(
    train_df['processed_text'].str.split().apply(len), 95))

int_vectorizer = TextVectorization(
    output_mode='int', max_tokens=MAX_TOKENS, output_sequence_length=sequence_length)
int_vectorizer.adapt(train_df['processed_text'].tolist())

label_encoder = LabelEncoder().fit(df['category'])
class_names = label_encoder.classes_

X_train = int_vectorizer(train_df['processed_text'].tolist()).numpy()
y_train = label_encoder.transform(train_df['category'])
X_test = int_vectorizer(test_df['processed_text'].tolist()).numpy()
y_test = label_encoder.transform(test_df['category'])
vocab_size = int_vectorizer.vocabulary_size()

# Lab 4 RNN baseline, for direct comparison in Task 5
rnn_baseline = {"short_query_accuracy": 0.881, "long_query_accuracy": 0.67}

print(f"Train/test split: {len(train_df)} / {len(test_df)} queries")
print(f"Sequence length (training cutoff): {sequence_length}")
print(f"Vocabulary size: {vocab_size}")

Train/test split: 336 / 84 queries
Sequence length (training cutoff): 7
Vocabulary size: 182


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data

In [34]:
def build_lstm_model(vocab_size: int, num_classes: int) -> keras.Model:
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM),
        LSTM(RNN_UNITS),
        Dense(num_classes, activation='softmax')])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    return model

lstm_model = build_lstm_model(vocab_size, num_classes=len(class_names))
lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
def build_gru_model(vocab_size: int, num_classes: int) -> keras.Model:
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM),
        GRU(RNN_UNITS),
        Dense(num_classes, activation='softmax')])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    return model

gru_model = build_gru_model(vocab_size, num_classes=len(class_names))
gru_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True
)

results = {}
for name, model in [("LSTM", lstm_model), ("GRU", gru_model)]:
    history = model.fit(
        X_train, y_train, validation_split=0.2,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[early_stop], verbose=0
    )
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    results[name] = acc
    print(f"{name}: stopped after {len(history.history['loss'])} epochs, "
          f"held-out test accuracy: {acc:.3f}")

print(f"\nSimple RNN (Lab 4 baseline): {rnn_baseline['short_query_accuracy']:.3f}")

LSTM: stopped after 9 epochs, held-out test accuracy: 0.964
GRU: stopped after 5 epochs, held-out test accuracy: 0.381

Simple RNN (Lab 4 baseline): 0.881


In [37]:

long_query_samples = [
    {"ticket_id": "SC-4831-01", "expected_category": "Electronics",
     "query": "looking for a lightweight wireless bluetooth headphone "
              "with good noise cancelling and fast charging under budget "
              "two thousand rupee price and good quality"},
    {"ticket_id": "SC-4831-02", "expected_category": "Electronics",
     "query": "need a smartphone with good camera fast charging under "
              "budget five thousand rupee price and best quality and "
              "good resolution"},
    {"ticket_id": "SC-4831-03", "expected_category": "Apparel",
     "query": "want a slim fit cotton casual shirt in navy blue and grey "
              "color with good size for men and good quality and stylish"},
    {"ticket_id": "SC-4831-04", "expected_category": "Apparel",
     "query": "looking for running shoe for woman with good fit and "
              "lightweight and stylish and good quality for casual wear"},
    {"ticket_id": "SC-4831-05", "expected_category": "Grocery",
     "query": "want organic basmati rice pack with good quality and "
              "fresh and natural and no cheap packaging and good price"},
    {"ticket_id": "SC-4831-06", "expected_category": "Grocery",
     "query": "looking for cold pressed olive oil bottle with good "
              "quality and fresh and natural and premium and good price"},
]


train_vocab = set(w for t in train_df['processed_text'] for w in t.split())
for s in long_query_samples:
    proc = preprocess(s['query']).split()
    known = [w for w in proc if w in train_vocab]
    print(f"{s['ticket_id']}: {len(known)}/{len(proc)} words in vocab "
          f"({len(known)/len(proc):.0%})")

eval_vectorizer = TextVectorization(output_mode='int', max_tokens=MAX_TOKENS)
eval_vectorizer.adapt(train_df['processed_text'].tolist())
def evaluate_on_queries(model, vectorizer, samples):
    texts = [preprocess(s['query']) for s in samples]
    sequences = vectorizer(texts).numpy()
    probs = model.predict(sequences, verbose=0)
    out = pd.DataFrame(samples)
    out['predicted_category'] = class_names[np.argmax(probs, axis=1)]
    out['confidence'] = probs.max(axis=1).round(2)
    return out

print()
for name, model in [("LSTM", lstm_model), ("GRU", gru_model)]:
    out = evaluate_on_queries(model, eval_vectorizer, long_query_samples)
    acc = (out['predicted_category'] == out['expected_category']).mean()
    print(f"--- {name} ---")
    print(out[['ticket_id', 'expected_category', 'predicted_category', 'confidence']])
    print(f"{name} accuracy on SC-4831 long queries: {acc:.2f}\n")

SC-4831-01: 15/17 words in vocab (88%)
SC-4831-02: 11/15 words in vocab (73%)
SC-4831-03: 14/16 words in vocab (88%)
SC-4831-04: 11/12 words in vocab (92%)
SC-4831-05: 12/13 words in vocab (92%)
SC-4831-06: 11/13 words in vocab (85%)

--- LSTM ---
    ticket_id expected_category predicted_category  confidence
0  SC-4831-01       Electronics        Electronics        0.34
1  SC-4831-02       Electronics        Electronics        0.34
2  SC-4831-03           Apparel            Apparel        0.38
3  SC-4831-04           Apparel            Apparel        0.35
4  SC-4831-05           Grocery            Grocery        0.34
5  SC-4831-06           Grocery            Grocery        0.35
LSTM accuracy on SC-4831 long queries: 1.00

--- GRU ---
    ticket_id expected_category predicted_category  confidence
0  SC-4831-01       Electronics            Grocery        0.34
1  SC-4831-02       Electronics        Electronics        0.35
2  SC-4831-03           Apparel        Electronics        0.34
3 

Lab no 6

In [38]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.layers import TextVectorization, Embedding, Dense, GlobalAveragePooling1D

# Step 1: load the dataset
df = pd.read_csv("smartcart_customer_queries.csv")
print(df.shape)
print(df['category'].value_counts())

# Step 2: turn each query's text into a fixed-length list of numbers
MAX_LEN = 15
vectorizer = TextVectorization(output_mode="int", output_sequence_length=MAX_LEN)
vectorizer.adapt(df['query_text'])
X = vectorizer(df['query_text']).numpy()

# Step 3: turn each category into a number
labels = df['category'].astype('category')
y = labels.cat.codes.to_numpy()
class_names = labels.cat.categories
vocab_size = vectorizer.vocabulary_size()

print("Vocabulary size:", vocab_size)
print("Categories:", list(class_names))

(420, 3)
category
Apparel        140
Grocery        140
Electronics    140
Name: count, dtype: int64
Vocabulary size: 209
Categories: ['Apparel', 'Electronics', 'Grocery']


In [40]:



# Step 1: a tiny model -- Embedding, then built-in Attention, then classify
inputs = keras.Input(shape=(MAX_LEN,))
embed = Embedding(input_dim=vocab_size, output_dim=16)(inputs)

attention_output, attention_scores = tf.keras.layers.Attention()(
    [embed, embed], return_attention_scores=True
)
pooled = GlobalAveragePooling1D()(attention_output)
outputs = Dense(len(class_names), activation="softmax")(pooled)

model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Step 2: a second "view" into the SAME model, that returns attention scores
# instead of the final prediction -- doesn't need separate training
attention_probe = keras.Model(inputs, attention_scores)

# Step 3: train the model
model.fit(X, y, epochs=15, batch_size=32, verbose=0)

# Step 4: pick a real, longer complaint query and clean it simply
sample_text = "NOT  SATISFIED WITH THE WINTER JACKET, FEELS CHEAP AND LOW QUALITY"
tokens = sample_text.lower().replace(",", "").split()
print("Tokens:", tokens)

# Step 5: look at what the LAST word attends to most
sample_seq = vectorizer([sample_text])
scores = attention_probe.predict(sample_seq, verbose=0)[0]
last_row = scores[len(tokens) - 1][:len(tokens)]

ranked = sorted(zip(tokens, last_row), key=lambda x: -x[1])
print("\nAttention weights for last word 'quality':")
for t, w in ranked:
    print(f"  {t:10s} {w:.3f}")

Tokens: ['not', 'satisfied', 'with', 'the', 'winter', 'jacket', 'feels', 'cheap', 'and', 'low', 'quality']

Attention weights for last word 'quality':
  feels      0.069
  quality    0.069
  low        0.068
  with       0.068
  the        0.068
  not        0.068
  satisfied  0.068
  cheap      0.066
  and        0.065
  winter     0.061
  jacket     0.061


In [42]:
# Step 1: no mask -> BERT-style (every word sees every other word)
attn_layer = tf.keras.layers.Attention()
embed_layer = model.get_layer(index=1)   # reuse the trained Embedding layer
sample_embeddings = embed_layer(sample_seq)

_, weights_bert = attn_layer([sample_embeddings, sample_embeddings],
                              return_attention_scores=True)

# Step 2: causal mask -> GPT-style (each word sees only itself and earlier words)
_, weights_gpt = attn_layer([sample_embeddings, sample_embeddings],
                             return_attention_scores=True, use_causal_mask=True)

weights_bert = weights_bert.numpy()[0]
weights_gpt = weights_gpt.numpy()[0]

jacket_idx = tokens.index("jacket")

print(f"{'word':10s} {'BERT':>7s} {'GPT':>7s}")
for i, t in enumerate(tokens):
    print(f"{t:10s} {weights_bert[jacket_idx, i]:7.3f} {weights_gpt[jacket_idx, i]:7.3f}")

print(f"\nWords visible to 'jacket': BERT sees all {len(tokens)}; GPT sees only {jacket_idx + 1}")

word          BERT     GPT
not          0.055   0.105
satisfied    0.060   0.116
with         0.059   0.112
the          0.053   0.101
winter       0.141   0.270
jacket       0.154   0.296
feels        0.056   0.000
cheap        0.045   0.000
and          0.044   0.000
low          0.058   0.000
quality      0.051   0.000

Words visible to 'jacket': BERT sees all 11; GPT sees only 6


In [43]:
from transformers import pipeline

# BERT's pretraining task -- fill in a masked word
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
masked_query = "not satisfied with the winter [MASK], feels cheap and low quality"
for r in fill_mask(masked_query)[:3]:
    print(f"{r['token_str']:>10s}  (score={r['score']:.3f})")

# GPT-2's pretraining task -- continue text one word at a time
generator = pipeline("text-generation", model="gpt2")
prompt = "Customer: The winter jacket I bought feels cheap.\nAssistant:"
result = generator(prompt, max_new_tokens=30, num_return_sequences=1)
print(result[0]['generated_text'])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

... (9 outputs left out)

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Customer: The winter jacket I bought feels cheap.
Assistant: The jacket looks like you used to wear it.
Assistant: It's really long.
Assistant: It's nice.
Assistant: The size
